<a href="https://colab.research.google.com/github/EllenKAlves/ProjetoAurora/blob/main/script_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Leitura dos Dados

In [267]:
import pandas as pd
import re

df = pd.read_csv('/Telemetria_relatorio - Página1.csv')
df.columns = df.columns.str.strip()
df_foguetes = df.set_index('Foguete').copy()

df_foguetes['temperatura_externa'] = df.set_index('Foguete')['temperatura_externa']
df_foguetes['temperatura_interna'] = df.set_index('Foguete')['temperatura_interna']

print("df_foguetes atualizado com as colunas de temperatura:")
display(df_foguetes.head())

def limpar_valor(texto):
    """Extrai apenas o primeiro número de uma string (ex: '~4.500' -> 4500)"""
    if pd.isna(texto): return None
    texto = str(texto).replace('.', '')
    match = re.search(r"(-?\d+(?:,\d+)?)", texto)
    if match:
        #Converte vírgula decimal para ponto se existir
        return float(match.group(1).replace(',', '.'))
    return None

df_foguetes atualizado com as colunas de temperatura:


,Capacidade (kWh eq.),Carga Atual,Empuxo total(kN),Consumo prop. (kg/s),Perdas Energ.,Tipo Press.,Tipo Propulsor,T queima 1º est.,temperatura_interna,temperatura_externa
Foguete,,,,,,,,,,
Falcon 9,~4.500,100%,7.607,~2.500,2-3%,He COPV,Turbobomba GG,~162s,20–25 °C,"-206°C(LOX)- -6,6°C(RP-1)"
New Glenn,~6.800,100%,17.000,~3.200,3-4%,Autógena,Staged Comb.,Staged Comb.,20–25 °C,"-183°C(LOX)- -6,6°C(LNG)- -253°C(LH2)"
Electron,~12,100%,162.000,~50,5-7%,He Press.,Bomba elétrica,~153s,15–30 °C,-183°C(LOX)- ~20°C(RP-1)
Terran R,~5.200,100%,14.900,~2.800,3-5%,Subcooled + He/N₂,GG alta press.,~170s(est),20–25 °C,-183°C(lOX)- -162°(CH4)


# Estatística Descritiva

In [268]:
df.describe()

,Empuxo total(kN)
count,4.000000
mean,50.376750
std,74.524288
min,7.607000
25%,13.076750
50%,15.950000
75%,53.250000
max,162.000000


# Verificações

In [269]:
limites = {
    "temp_interna": (15, 30),        # faixa segura
    "temp_externa": (-220, -140),    # criogênico
    "energia_min": 20,               # %
    "pressao_min": 5                 # bar (adaptado)
}

# Criando Alertas

In [270]:
import pandas as pd
import re

#Função de suporte para limpar os dados (tira %, °C e ~)
def extrair_numero(texto):
    if pd.isna(texto): return 0.0
    texto_limpo = str(texto).replace('.', '')
    match = re.search(r"(-?\d+(?:[.,]\d+)?)", texto_limpo)
    if match:
        return float(match.group(1).replace(',', '.'))
    return 0.0

#Loop principal (Percorre a planilha linha por linha)
print("🔍 Iniciando Verificação de Alertas de Telemetria...\n")

for _, linha in df.iterrows():
    foguete = linha['Foguete']

    #MAPEAMENTO DAS COLUNAS DA SUA PLANILHA
    nivel_energia = extrair_numero(linha['Carga Atual'])
    temp_int = extrair_numero(linha['temperatura_interna'])
    temp_ext = extrair_numero(linha['temperatura_externa'])

    print(f"🚀 Verificando Veículo: {foguete}")

    # Alerta de Energia (Baseado na Carga Atual)
    if nivel_energia < 100:
        print(f"ALERTA ENERGIA: Bateria em {nivel_energia}% - Requer Carga.")
    else:
        print(f"Energia: Estável ({nivel_energia}%)")

    # Alerta de Temperatura Interna
    if temp_int > 25:
        print(f"ALERTA CALOR: Interna em {temp_int}°C - Resfriamento Ativado.")
    elif temp_int < 20:
        print(f"ALERTA FRIO: Interna em {temp_int}°C - Aquecimento Ativado.")
    else:
        print(f"Temp. Interna: Nominal ({temp_int}°C)")

    # Alerta de Temperatura Externa
    if temp_ext < -200:
        print(f"CRÍTICO: Temp. Externa Extremamente Baixa ({temp_ext}°C)!")
    else:
        print(f"Temp. Externa: Dentro dos Parâmetros ({temp_ext}°C)")

    print("-" * 50)

🔍 Iniciando Verificação de Alertas de Telemetria...

🚀 Verificando Veículo: Falcon 9
Energia: Estável (100.0%)
Temp. Interna: Nominal (20.0°C)
CRÍTICO: Temp. Externa Extremamente Baixa (-206.0°C)!
--------------------------------------------------
🚀 Verificando Veículo: New Glenn
Energia: Estável (100.0%)
Temp. Interna: Nominal (20.0°C)
Temp. Externa: Dentro dos Parâmetros (-183.0°C)
--------------------------------------------------
🚀 Verificando Veículo: Electron
Energia: Estável (100.0%)
ALERTA FRIO: Interna em 15.0°C - Aquecimento Ativado.
Temp. Externa: Dentro dos Parâmetros (-183.0°C)
--------------------------------------------------
🚀 Verificando Veículo: Terran R
Energia: Estável (100.0%)
Temp. Interna: Nominal (20.0°C)
Temp. Externa: Dentro dos Parâmetros (-183.0°C)
--------------------------------------------------


# Processamento

In [271]:
import pandas as pd
import re

# Redefinindo a função limpar_valor, caso ainda não esteja definida ou para garantir consistência.
def limpar_valor(texto):
    """Extrai apenas o primeiro número de uma string (ex: '~4.500' -> 4500)"""
    if pd.isna(texto): return None
    texto = str(texto).replace('.', '')
    match = re.search(r"(-?\d+(?:,\d+)?)", texto)
    if match:
        #Converte vírgula decimal para ponto se existir
        return float(match.group(1).replace(',', '.'))
    return None

def analisar_linha(linha):
    alertas = []

    # Usar a função limpar_valor para garantir que os dados sejam numéricos
    temp_int = limpar_valor(linha['temperatura_interna'])
    temp_ext = limpar_valor(linha['temperatura_externa'])
    nivel_energia = limpar_valor(linha['Carga Atual'])

    # Verificar se os valores foram extraídos com sucesso
    if temp_int is None or temp_ext is None or nivel_energia is None:
        return alertas # Retorna sem alertas se os dados não são válidos para análise

    # ALERTA DE TEMPERATURA INTERNA
    if not (15 <= temp_int <= 28):
        alertas.append("TEMP_INTERNA_FORA_IDEAL")

    # ALERTA DE TEMPERATURA EXTERNA CRÍTICA
    if temp_ext < -200:
        alertas.append("TEMP_EXTERNA_CRITICA")

    # Alerta de Energia (mantendo o limite de 20% para 'MISSÃO EM RISCO')
    if nivel_energia < limites["energia_min"]:
        alertas.append("ENERGIA_BAIXA")

    return alertas

def processar(df):
    total_alertas = 0
    eventos_criticos = 0

    for i, linha in df.iterrows():
        alertas = analisar_linha(linha)

        if alertas:
            total_alertas += len(alertas)

            # Se tiver falha estrutural ou navegação → crítico
            if "FALHA_ESTRUTURAL" in alertas or "FALHA_NAVEGACAO" in alertas:
                eventos_criticos += 1

    return total_alertas, eventos_criticos

# Verificação Final



In [272]:
print("🔍 Iniciando Verificação de Alertas e Status da Missão...\n")

# Loop para exibir alertas individuais e fazer uma contagem simples
for foguete, linha in df_foguetes.iterrows():

    nivel_energia = limpar_valor(linha['Carga Atual'])
    temp_int = limpar_valor(linha['temperatura_interna'])
    temp_ext = limpar_valor(linha['temperatura_externa'])

    if nivel_energia is None or temp_int is None or temp_ext is None:
        print(f"🚀 ERRO: Dados inválidos para o foguete {foguete}. Pulando análise.")
        print("-" * 40)
        continue

    print(f"🚀 Analisando Veículo: {foguete}")

    # Alerta de Energia
    if nivel_energia < 100:
        print(f"ALERTA ENERGIA: Carga em {nivel_energia}% (Revisar painéis solares/bateria)")
    else:
        print(f"Energia Estável: {nivel_energia}%")

    # Alerta de Temperatura Interna (usando os limites do código original 18 e 28)
    if not (18 <= temp_int <= 28):
        print(f"ALERTA T. INTERNA: {temp_int}°C fora da faixa ideal!")
    else:
        print(f"Temperatura Interna OK: {temp_int}°C")

    # Alerta de Temperatura Externa
    if temp_ext < -200:
        print(f"CRÍTICO: Temperatura Externa de {temp_ext}°C (Risco de congelamento de válvulas)")
    else:
        print(f"Temperatura Externa: Dentro dos Parâmetros ({temp_ext}°C)")

    alertas_foguete = analisar_linha(linha)
    if len(alertas_foguete) > 0:
        print(f"⚠️ MISSÃO EM RISCO para {foguete} devido a: {', '.join(alertas_foguete)}")
    else:
        print(f"✅ MISSÃO ESTÁVEL para {foguete}")

    print("-" * 40)

🔍 Iniciando Verificação de Alertas e Status da Missão...

🚀 Analisando Veículo: Falcon 9
Energia Estável: 100.0%
Temperatura Interna OK: 20.0°C
CRÍTICO: Temperatura Externa de -206.0°C (Risco de congelamento de válvulas)
⚠️ MISSÃO EM RISCO para Falcon 9 devido a: TEMP_EXTERNA_CRITICA
----------------------------------------
🚀 Analisando Veículo: New Glenn
Energia Estável: 100.0%
Temperatura Interna OK: 20.0°C
Temperatura Externa: Dentro dos Parâmetros (-183.0°C)
✅ MISSÃO ESTÁVEL para New Glenn
----------------------------------------
🚀 Analisando Veículo: Electron
Energia Estável: 100.0%
ALERTA T. INTERNA: 15.0°C fora da faixa ideal!
Temperatura Externa: Dentro dos Parâmetros (-183.0°C)
✅ MISSÃO ESTÁVEL para Electron
----------------------------------------
🚀 Analisando Veículo: Terran R
Energia Estável: 100.0%
Temperatura Interna OK: 20.0°C
Temperatura Externa: Dentro dos Parâmetros (-183.0°C)
✅ MISSÃO ESTÁVEL para Terran R
----------------------------------------
